## Обучение модели для FJSPT с использованием подхода RL4CO (алгоритм PPO)

In [1]:
from pytorch_lightning.loggers import CometLogger
import torch

from lightning.pytorch.callbacks import ModelCheckpoint, RichModelSummary

from env import FJSPTEnv
from rl4co.utils.trainer import RL4COTrainer

from model import L2DPPOModel
from l2d_policy import L2DPolicy4PPO

In [2]:
from rl4co.envs import ENV_REGISTRY

ENV_REGISTRY["fjspt"] = FJSPTEnv

env = FJSPTEnv(
    generator_params={
      "num_jobs": 20,  # the total number of jobs
      "num_machines": 10,  # the total number of machines that can process operations
      "num_trucks": 2,  # the total number of trucks
      "min_ops_per_job": 15,  # minimum number of operatios per job
      "max_ops_per_job": 25,  # maximum number of operations per job
      "min_processing_time": 20,  # the minimum time required for a machine to process an operation
      "max_processing_time": 70,  # the maximum time required for a machine to process an operation
      "min_transportation_time": 0,  # the minimum time required for a truck to transport
      "max_transportation_time": 50,  # the maximum time required for a truck to transport
      "min_eligible_ma_per_op": 1,  # the minimum number of machines capable to process an operation
      "max_eligible_ma_per_op": 6,  # the maximum number of machines capable to process an operation
    },
)

In [3]:
accelerator = "gpu"
batch_size = 1024
max_epochs = 100
train_data_size = batch_size * 100
val_data_size = 1_000
test_data_size = 1_000
embed_dim = 128
num_encoder_layers = 4

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# Политика диспетчеризации (Learning to Dispatch):
# нейросеть encoder–decoder, выбирающая следующее допустимое назначение операции на машину и транспортное средство
policy = L2DPolicy4PPO(embed_dim=embed_dim, num_encoder_layers=num_encoder_layers)
policy = policy.to(device)

# RL-модель Learning to Dispatch:
# обучает политику методом REINFORCE с rollout-бейзлайном,
# оптимизируя makespan через взаимодействие с FJSP-симуляцией
model = L2DPPOModel(env,
                 policy=policy, 
                 batch_size=batch_size,
                 train_data_size=train_data_size,
                 val_data_size=val_data_size,
                 test_data_size=test_data_size,
                 optimizer_kwargs={"lr": 1e-4})

In [ ]:
policy.eval()
td = env.reset(batch_size=[1]).to(device)
td1 = td.clone()

with torch.no_grad():
    out = policy(td1, env, phase="test", decode_type="greedy", return_actions=True)

## Обучение

In [5]:
policy.train()

L2DPolicy4PPO(
  (encoder): HetGNNEncoder(
    (init_embedding): FJSPTInitEmbedding(
      (init_ops_embed): Linear(in_features=5, out_features=128, bias=False)
      (pos_encoder): PositionalEncoding(
        (dropout): Dropout(p=0.0, inplace=False)
      )
      (init_ma_embed): Linear(in_features=1, out_features=128, bias=False)
      (init_truck_embed): Linear(in_features=1, out_features=128, bias=False)
      (proc_edge_embed): Linear(in_features=1, out_features=128, bias=False)
      (truck_edge_embed): Linear(in_features=1, out_features=128, bias=False)
    )
    (layers): ModuleList(
      (0-3): 4 x HetGNNBlock(
        (ma_from_ops): HetGNNLayer(
          (activation): ReLU()
        )
        (ops_from_ma): HetGNNLayer(
          (activation): ReLU()
        )
        (ma_from_ma): HetGNNLayer(
          (activation): ReLU()
        )
        (truck_from_ma): HetGNNLayer(
          (activation): ReLU()
        )
        (ffn_ma): TransformerFFN(
          (ops): ModuleDict(

In [6]:
checkpoint_callback = ModelCheckpoint(
    dirpath="checkpoints",
    filename="ppo_epoch_{epoch:03d}",
    save_top_k=1,
    save_last=True,
    monitor="val/reward",
    mode="max",  # maximize reward = minimize makespan
)

rich_model_summary = RichModelSummary(max_depth=3)

callbacks = [checkpoint_callback, rich_model_summary]

In [7]:
from comet_key import API_KEY

logger = CometLogger(
    api_key=API_KEY,
    project="rl4co_ppo",
)

trainer = RL4COTrainer(
    max_epochs=max_epochs,
    accelerator=accelerator,
    devices=1,
    logger=logger,
    callbacks=callbacks,
)

COMET WARNING: To get all data logged automatically, import comet_ml before the following modules: torch.
COMET WARNING: As you are running in a Jupyter environment, you will need to call `experiment.end()` when finished to ensure all metrics and code are logged before exiting.
COMET INFO: Experiment is live on comet.com https://www.comet.com/fsoidhfoi/rl4co-ppo/7e6a8b7faf7f46c2826f8767a9b81544

Using 16bit Automatic Mixed Precision (AMP)
Trainer already configured with model summary callbacks: [<class 'lightning.pytorch.callbacks.rich_model_summary.RichModelSummary'>]. Skipping setting a default `ModelSummary` callback.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


In [7]:
trainer = RL4COTrainer(
    max_epochs=max_epochs,
    accelerator=accelerator,
    devices=1,
    logger=None,
    callbacks=callbacks,
)

Using 16bit Automatic Mixed Precision (AMP)
Trainer already configured with model summary callbacks: [<class 'lightning.pytorch.callbacks.rich_model_summary.RichModelSummary'>]. Skipping setting a default `ModelSummary` callback.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
/home/egor/cousework/rl4co_try/.venv/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/logger_connector/logger_connector.py:76: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `lightning.pytorch` package, due to potential conflicts with other packages in the ML ecosystem. For this reason, `logger=True` will use `CSVLogger` as the default logger, unless the `tensorboard` or `tensorboardX` packages are found. Please `pip install lightning[extra]` or one of them to enable TensorBoard support by default
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogg

In [ ]:
trainer.fit(model)

Overriding gradient_clip_val to None for 'automatic_optimization=False' models
val_file not set. Generating dataset instead
test_file not set. Generating dataset instead
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/home/egor/cousework/rl4co_try/.venv/lib/python3.12/site-packages/lightning/pytorch/utilities/model_summary/model_summary.py:242: Precision 16-mixed is not supported by the model summary.  Estimated model size in MB will not be accurate. Using 32 bits instead.


┏━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃    ┃ Name                                 ┃ Type               ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0  │ env                                  │ FJSPTEnv           │      0 │ train │     0 │
│ 1  │ policy                               │ L2DPolicy4PPO      │  1.7 M │ train │     0 │
│ 2  │ policy.encoder                       │ HetGNNEncoder      │  804 K │ train │     0 │
│ 3  │ policy.encoder.init_embedding        │ FJSPTInitEmbedding │  1.2 K │ train │     0 │
│ 4  │ policy.encoder.layers                │ ModuleList         │  803 K │ train │     0 │
│ 5  │ policy.decoder                       │ L2DDecoder         │  870 K │ train │     0 │
│ 6  │ policy.decoder.feature_extractor     │ HetGNNEncoder      │  804 K │ train │     0 │
│ 7  │ policy.decoder.actor                 │ FJSPTActor         │ 66.3 K │ train │     0 │
│ 8  │ policy.critic                        │ MLP                │ 65.9 K │ train │     0 │
│ 9  │ policy.critic.hidden_act             │ ReLU               │      0 │ train │     0 │
│ 10 │ policy.critic.out_act                │ Identity           │      0 │ train │     0 │
│ 11 │ policy.critic.lins                   │ ModuleList         │ 65.9 K │ train │     0 │
│ 12 │ policy.critic.input_norm             │ Identity           │      0 │ train │     0 │
│ 13 │ policy.critic.output_norm            │ Identity           │      0 │ train │     0 │
│ 14 │ policy_old                           │ L2DPolicy4PPO      │  1.7 M │ train │     0 │
│ 15 │ policy_old.encoder                   │ HetGNNEncoder      │  804 K │ train │     0 │
│ 16 │ policy_old.encoder.init_embedding    │ FJSPTInitEmbedding │  1.2 K │ train │     0 │
│ 17 │ policy_old.encoder.layers            │ ModuleList         │  803 K │ train │     0 │
│ 18 │ policy_old.decoder                   │ L2DDecoder         │  870 K │ train │     0 │
│ 19 │ policy_old.decoder.feature_extractor │ HetGNNEncoder      │  804 K │ train │     0 │
│ 20 │ policy_old.decoder.actor             │ FJSPTActor         │ 66.3 K │ train │     0 │
│ 21 │ policy_old.critic                    │ MLP                │ 65.9 K │ train │     0 │
│ 22 │ policy_old.critic.hidden_act         │ ReLU               │      0 │ train │     0 │
│ 23 │ policy_old.critic.out_act            │ Identity           │      0 │ train │     0 │
│ 24 │ policy_old.critic.lins               │ ModuleList         │ 65.9 K │ train │     0 │
│ 25 │ policy_old.critic.input_norm         │ Identity           │      0 │ train │     0 │
│ 26 │ policy_old.critic.output_norm        │ Identity           │      0 │ train │     0 │
└────┴──────────────────────────────────────┴────────────────────┴────────┴───────┴───────┘

Trainable params: 3.5 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 3.5 M                                                                                                
Total estimated model params size (MB): 13                                                                         
Modules in train mode: 707                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

/home/egor/cousework/rl4co_try/.venv/lib/python3.12/site-packages/lightning/pytorch/utilities/_pytree.py:21: 
`isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` 
instead.

...

predecessor indices = tensor([256], device='cuda:0')

row[idx] = tensor([1.], device='cuda:0')

times[idx] = tensor([933.], device='cuda:0')

products = tensor([933.], device='cuda:0')

sum = tensor(933., device='cuda:0')

## Тестирование обученной модели

In [ ]:
trainer.test(model)